<a href="https://colab.research.google.com/github/gouravbembi/Ip-assignments/blob/main/Assignment8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)

if img is None:
    raise ValueError("Image could not be loaded!")

img = img.astype(np.float32)
M, N = img.shape

a = 15
mean = 0
sigma = 10
k = 3
lam = 0.1
lambda_exp = 0.1
rayleigh_sigma = 10

u = np.random.rand(M, N)
uniform_noise = -a + (2 * a * u)

u1 = np.random.rand(M, N)
u2 = np.random.rand(M, N)

gaussian_noise = (
    sigma * np.sqrt(-2 * np.log(u1)) *
    np.cos(2 * np.pi * u2) + mean
)

erlang_noise = np.zeros((M, N))

for i in range(k):
    u = np.random.rand(M, N)
    erlang_noise += -np.log(u) / lam

erlang_noise -= k / lam

u = np.random.rand(M, N)
exponential_noise = -np.log(1 - u) / lambda_exp
exponential_noise -= 1 / lambda_exp

u = np.random.rand(M, N)

rayleigh_noise = rayleigh_sigma * np.sqrt(
    -2 * np.log(1 - u)
)

rayleigh_noise -= rayleigh_sigma * np.sqrt(np.pi / 2)

def add_noise(image, noise):
    return np.clip(image + noise, 0, 255).astype(np.uint8)

uniform_img = add_noise(img, uniform_noise)
gaussian_img = add_noise(img, gaussian_noise)
erlang_img = add_noise(img, erlang_noise)
exponential_img = add_noise(img, exponential_noise)
rayleigh_img = add_noise(img, rayleigh_noise)

uniform_builtin = np.random.uniform(-a, a, (M, N))

gaussian_builtin = np.random.normal(
    mean, sigma, (M, N)
)

erlang_builtin = (
    np.random.gamma(k, 1 / lam, (M, N)) - k / lam
)

exponential_builtin = (
    np.random.exponential(1 / lambda_exp, (M, N))
    - 1 / lambda_exp
)

rayleigh_builtin = (
    np.random.rayleigh(rayleigh_sigma, (M, N))
    - rayleigh_sigma * np.sqrt(np.pi / 2)
)

images = [
    img, uniform_img, gaussian_img,
    erlang_img, exponential_img, rayleigh_img
]

titles = [
    "Original Image", "Uniform Noise",
    "Gaussian Noise", "Erlang Noise",
    "Exponential Noise", "Rayleigh Noise"
]

plt.figure(figsize=(15, 9))

for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i], cmap='gray', vmin=0, vmax=255)
    plt.title(titles[i])
    plt.axis("off")

plt.tight_layout()
plt.show()

custom_noises = [
    uniform_noise, gaussian_noise, erlang_noise,
    exponential_noise, rayleigh_noise
]

builtin_noises = [
    uniform_builtin, gaussian_builtin, erlang_builtin,
    exponential_builtin, rayleigh_builtin
]

noise_names = [
    "Uniform", "Gaussian", "Erlang",
    "Exponential", "Rayleigh"
]

plt.figure(figsize=(15, 15))

for i in range(5):
    plt.subplot(5, 2, 2*i + 1)
    plt.hist(custom_noises[i].flatten(), bins=60, density=True)
    plt.title(noise_names[i] + " Noise (PDF Equation)")
    plt.xlabel("Noise Value")
    plt.ylabel("Probability Density")
    plt.grid()

    plt.subplot(5, 2, 2*i + 2)
    plt.hist(builtin_noises[i].flatten(), bins=60, density=True)
    plt.title(noise_names[i] + " Noise (Built-in Function)")
    plt.xlabel("Noise Value")
    plt.ylabel("Probability Density")
    plt.grid()

plt.tight_layout()
plt.show()

print("NOISE STATISTICS\n")

for name, noise in zip(noise_names, custom_noises):
    print(name, "Noise:")
    print("Mean:", np.mean(noise))
    print("Variance:", np.var(noise))
    print("Standard Deviation:", np.std(noise))
    print("--------------------------------")
